# 05 · Condition any study on the market regime

> Goal: split research by the SPY volatility regime (low / mid / high).
> Factor behavior is regime-dependent — pooling all dates muddies any
> signal.

## What you'll see

1. Today's regime, from `/market-context`
2. The full regime history (HOBBY+ unlocks this)
3. A pattern for splitting any per-ticker study by regime

In [1]:
# Setup — works with or without an API key.
# With FACTORWEAVE_API_KEY set, we use the full API (10,000+ tickers).
# Without one, we fall back to /demo/{ticker} (AAPL, MSFT, NVDA, AMZN, GOOGL, META, TSLA, JPM).
import os, json
import requests

API_BASE = "https://factorweave.com/api"
API_KEY = os.environ.get("FACTORWEAVE_API_KEY")
DEMO_TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "JPM"]
MODE = "live" if API_KEY else "demo"
print(f"Running in {MODE} mode.", "Key prefix:", (API_KEY[:8] + '…') if API_KEY else "(none)")


def fw_demo(ticker: str) -> dict:
    """Demo endpoint — no auth, 8 sample tickers, current snapshot only."""
    r = requests.get(f"{API_BASE}/demo/{ticker}", timeout=10)
    r.raise_for_status()
    return r.json()


def fw_features(ticker: str, **kwargs) -> dict:
    """Authed features endpoint when a key is available; demo fallback otherwise."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/features/{ticker}",
                         params=kwargs,
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — reshape demo response to look like the authed one
    d = fw_demo(ticker)
    return {"rows": [{"ticker": d["ticker"], "date": d["as_of"], **d["factors"]}]}


def fw_top(factor: str, n: int = 25) -> dict:
    """Top-N by a factor. Needs auth for the full universe; in demo mode we
    rank the 8 demo tickers locally."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/top",
                         params={"factor": factor, "n": n},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — fetch each demo ticker, sort locally
    rows = []
    for t in DEMO_TICKERS:
        d = fw_demo(t)
        if factor in d["factors"]:
            rows.append({"ticker": t, "date": d["as_of"], factor: d["factors"][factor]})
    rows.sort(key=lambda r: r[factor], reverse=True)
    return {"rows": rows[:n]}


def fw_similar(ticker: str, method: str = "cosine", limit: int = 10) -> dict:
    """Similarity search. Demo endpoint includes pre-computed `similar` set."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/vector-search/similar/{ticker}",
                         params={"method": method, "limit": limit, "min_lookback_days": 30},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — uses the `similar` array baked into the demo response
    d = fw_demo(ticker)
    return {"ticker": ticker, "method": "cosine (demo)", "neighbors": d.get("similar", [])[:limit]}


def fw_market_context() -> dict:
    """Universe analytics. Public on FREE, fuller on HOBBY+."""
    headers = {"X-API-Key": API_KEY} if API_KEY else {}
    r = requests.get(f"{API_BASE}/market-context", params={"latest": 1}, headers=headers, timeout=10)
    if r.status_code == 401:
        return {"_note": "market-context requires auth in demo mode"}
    r.raise_for_status()
    return r.json()


Running in demo mode. Key prefix: (none)


## Today's regime

In [2]:
ctx = fw_market_context()
print("regime:", ctx.get("regime", "(not available without auth)"))
print("as of: ", ctx.get("date", "—"))

# Show whatever fields the response carries
for k, v in (ctx or {}).items():
    if k in ("regime", "date"): continue
    if isinstance(v, (dict, list)):
        print(f"{k}: ({type(v).__name__}, {len(v)} entries)")
    else:
        print(f"{k}: {v}")

regime: (not available without auth)
as of:  —
_note: market-context requires auth in demo mode


## Full regime history (HOBBY+)

In [3]:
import requests, pandas as pd

if not API_KEY:
    print("Set FACTORWEAVE_API_KEY to fetch the full regime history.")
    history = None
else:
    r = requests.get(f"{API_BASE}/market-context",
                     headers={"X-API-Key": API_KEY},
                     timeout=15)
    if r.status_code == 403:
        print("Full history is HOBBY+ gated.")
        history = None
    else:
        r.raise_for_status()
        history = r.json()
        print("Got", len(history.get("history", [])), "rows")
        if history.get("history"):
            df_reg = pd.DataFrame(history["history"])
            print(df_reg.tail())

Set FACTORWEAVE_API_KEY to fetch the full regime history.


## The regime-split pattern

For any analysis (a screen, a backtest, a similarity sweep), the pattern is:

```python
# 1. attach the regime to your per-date rows
df_with_regime = df.merge(df_regime[["date", "regime"]], on="date")

# 2. split + summarize
for regime, sub in df_with_regime.groupby("regime"):
    print(regime, "n=", len(sub),
          "mean fwd_ret_20d:", sub["fwd_ret_20d"].mean())
```

Common findings:
- Momentum tends to work better in trending (low/mid-vol) regimes
- Mean-reversion tends to work better in high-vol regimes
- Composite scores are noisier in regime transitions than within stable regimes

> All "tends to" disclaimers apply — see the [research note](https://factorweave.com/research.html) for what we've actually measured.

## You've reached the end of the tour

You now know how to:
- Pull factor data (`01`)
- Screen + filter (`02`)
- Find factor-similar setups (`03`)
- Assemble leak-free backtests (`04`)
- Split work by market regime (`05`)

Everything else in the API surface is a variation on these patterns. See the
[full docs](https://factorweave.com/#docs) and the [OpenAPI spec](https://factorweave.com/api/openapi.json) for the complete endpoint reference.